# Notebook 12 — Final Model Selection and Manuscript-Ready Performance Summary

## Objective
Summarize all completed modeling stages, compare clinical-only, clinical + DaTSCAN, and clinical + DaTSCAN + biomarker models, and prepare manuscript-ready tables/figures for scientific interpretation.

## Scientific Background
This notebook does **not** train new models. It summarizes the internal validation results generated in earlier notebooks and supports the final decision about whether biomarker and imaging features provide meaningful incremental value over clinical predictors for predicting Parkinson’s disease motor progression.

## Dataset Verification
This notebook assumes that Notebooks 05, 06, 09, and 11 have already been executed and their output folders are available in Google Drive.

## Expected Output
- Consolidated performance table
- Incremental performance summary
- Model selection decision table
- Manuscript-ready figures
- Quality control checklist
- Summary report


In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB05_DIR = PROJECT_DIR / "outputs" / "notebook_05_baseline_ml"
NB06_DIR = PROJECT_DIR / "outputs" / "notebook_06_model_refinement"
NB09_DIR = PROJECT_DIR / "outputs" / "notebook_09_clinical_vs_multimodal_models"
NB11_DIR = PROJECT_DIR / "outputs" / "notebook_11_biomarker_model_comparison"

OUT_DIR = PROJECT_DIR / "outputs" / "notebook_12_final_model_summary"
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR:", OUT_DIR)

for name, path in {
    "Notebook 05": NB05_DIR,
    "Notebook 06": NB06_DIR,
    "Notebook 09": NB09_DIR,
    "Notebook 11": NB11_DIR,
}.items():
    print(f"{name} exists:", path.exists(), "|", path)


## Code — Helper functions

In [ ]:
# ============================================================
# 03. Helper functions
# ============================================================

def safe_read_csv(path, label=None):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        df["source_file"] = path.name
        if label is not None:
            df["source_notebook"] = label
        return df
    print("Missing file:", path)
    return pd.DataFrame()

def add_analysis_label(df, label):
    if df.empty:
        return df
    df = df.copy()
    df["analysis_stage"] = label
    return df

def clean_performance_table(df):
    if df.empty:
        return df
    wanted_cols = [
        "analysis_stage", "feature_set", "model", "roc_auc", "pr_auc",
        "balanced_accuracy", "sensitivity", "specificity", "precision",
        "f1", "brier_score", "threshold", "tn", "fp", "fn", "tp",
        "n_test", "n_predictors", "evaluation"
    ]
    existing = [c for c in wanted_cols if c in df.columns]
    return df[existing].copy()

def round_numeric(df, digits=4):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].round(digits)
    return df


## Code — Load performance outputs from previous notebooks

In [ ]:
# ============================================================
# 04. Load model performance tables
# ============================================================

tables = []

# Notebook 05
nb05_test = safe_read_csv(NB05_DIR / "04_test_performance_primary_models.csv", "Notebook05")
nb05_test = add_analysis_label(nb05_test, "Clinical-only baseline models")
tables.append(nb05_test)

# Notebook 06
nb06_test = safe_read_csv(NB06_DIR / "04_tuned_test_performance.csv", "Notebook06")
nb06_test = add_analysis_label(nb06_test, "Tuned clinical-only models")
tables.append(nb06_test)

nb06_cal = safe_read_csv(NB06_DIR / "07_calibrated_test_performance.csv", "Notebook06")
nb06_cal = add_analysis_label(nb06_cal, "Calibrated clinical-only models")
tables.append(nb06_cal)

# Notebook 09
nb09_test = safe_read_csv(NB09_DIR / "04_test_performance_feature_set_models.csv", "Notebook09")
nb09_test = add_analysis_label(nb09_test, "Clinical vs DaTSCAN models")
tables.append(nb09_test)

# Notebook 11
nb11_test = safe_read_csv(NB11_DIR / "04_test_performance_biomarker_comparison.csv", "Notebook11")
nb11_test = add_analysis_label(nb11_test, "Clinical + DaTSCAN + biomarker models")
tables.append(nb11_test)

all_test = pd.concat([t for t in tables if not t.empty], ignore_index=True)
all_test_clean = clean_performance_table(all_test)
all_test_clean = round_numeric(all_test_clean)

print("Combined test performance table shape:", all_test_clean.shape)
display(all_test_clean.head(20))

all_test_clean.to_csv(OUT_DIR / "01_consolidated_test_performance_all_models.csv", index=False)


## Code — Load incremental performance summaries

In [ ]:
# ============================================================
# 05. Load incremental performance summaries
# ============================================================

incremental_tables = []

nb09_increment = safe_read_csv(NB09_DIR / "08_multimodal_incremental_performance_summary.csv", "Notebook09")
if not nb09_increment.empty:
    nb09_increment["increment_type"] = "DaTSCAN added to clinical predictors"
    incremental_tables.append(nb09_increment)

nb11_increment = safe_read_csv(NB11_DIR / "06_biomarker_incremental_performance_summary.csv", "Notebook11")
if not nb11_increment.empty:
    nb11_increment["increment_type"] = "SAA biomarkers added to clinical + DaTSCAN"
    incremental_tables.append(nb11_increment)

if incremental_tables:
    incremental_summary = pd.concat(incremental_tables, ignore_index=True)
else:
    incremental_summary = pd.DataFrame()

incremental_summary = round_numeric(incremental_summary)
display(incremental_summary)

incremental_summary.to_csv(OUT_DIR / "02_incremental_performance_summary.csv", index=False)


## Code — Identify top-performing models by selected metrics

In [ ]:
# ============================================================
# 06. Top-performing models by major metrics
# ============================================================

ranking_metrics = ["roc_auc", "pr_auc", "balanced_accuracy", "sensitivity", "specificity", "f1"]

top_rows = []
for metric in ranking_metrics:
    if metric in all_test_clean.columns and not all_test_clean.empty:
        temp = all_test_clean.dropna(subset=[metric]).sort_values(metric, ascending=False)
        if len(temp) > 0:
            row = temp.iloc[0].copy()
            row["selection_metric"] = metric
            row["selection_metric_value"] = row[metric]
            top_rows.append(row)

top_models_by_metric = pd.DataFrame(top_rows)
top_models_by_metric = round_numeric(top_models_by_metric)

display(top_models_by_metric[[
    "selection_metric", "selection_metric_value", "analysis_stage",
    "feature_set", "model", "roc_auc", "pr_auc", "balanced_accuracy",
    "sensitivity", "specificity", "f1"
]])

top_models_by_metric.to_csv(OUT_DIR / "03_top_models_by_metric.csv", index=False)


## Code — Final model decision logic

The decision rule used here is conservative:
1. Do not select a final model only because it performs best on the held-out test set.
2. Treat biomarkers as retained for exploratory analysis only if they improve ROC-AUC without major loss of PR-AUC or balanced accuracy.
3. If incremental gains are small and unstable, keep the simpler clinical or clinical + imaging model as the primary analysis and report biomarker models as secondary/exploratory.


In [ ]:
# ============================================================
# 07. Final model decision table
# ============================================================

# Extract useful deltas where available
delta_datscan = {}
delta_biomarker = {}

if not incremental_summary.empty:
    for _, row in incremental_summary.iterrows():
        inc_type = str(row.get("increment_type", ""))
        row_dict = row.to_dict()
        if "DaTSCAN" in inc_type:
            delta_datscan = row_dict
        if "SAA biomarkers" in inc_type:
            delta_biomarker = row_dict

decision_rows = []

decision_rows.append({
    "decision_domain": "Primary scientific conclusion",
    "decision": "Prediction performance is modest; models should not be presented as clinically deployable.",
    "rationale": "Best held-out ROC-AUC values remain close to 0.60 and class sensitivity/specificity trade-offs are limited."
})

decision_rows.append({
    "decision_domain": "Clinical-only model",
    "decision": "Retain as primary baseline comparator.",
    "rationale": "It is simpler, reproducible, and provides the reference for incremental value analyses."
})

decision_rows.append({
    "decision_domain": "DaTSCAN/SBR features",
    "decision": "Retain as secondary multimodal analysis.",
    "rationale": "DaTSCAN features produced only small incremental ROC-AUC changes and should be reported as exploratory/secondary."
})

decision_rows.append({
    "decision_domain": "SAA biomarker features",
    "decision": "Do not promote as final primary model; report as exploratory.",
    "rationale": "SAA biomarkers improved some sensitivity/F1 trade-offs but decreased PR-AUC and specificity in held-out evaluation."
})

decision_rows.append({
    "decision_domain": "Next analysis",
    "decision": "Prepare manuscript-ready tables and transparent limitations rather than further tuning on the same test set.",
    "rationale": "Repeated tuning against the same held-out test set risks overfitting the evaluation process."
})

final_decision = pd.DataFrame(decision_rows)
display(final_decision)

final_decision.to_csv(OUT_DIR / "04_final_model_selection_decision_table.csv", index=False)


## Code — Manuscript-ready compact performance table

In [ ]:
# ============================================================
# 08. Manuscript-ready compact performance table
# ============================================================

# Keep selected clinically meaningful rows:
# - best clinical-only/tuned rows
# - clinical + DaTSCAN rows
# - clinical + DaTSCAN + biomarker rows

compact = all_test_clean.copy()

# Prefer non-dummy models only
if "model" in compact.columns:
    compact = compact[~compact["model"].astype(str).str.contains("Dummy", case=False, na=False)].copy()

# Sort by ROC-AUC and keep top rows while preserving model diversity
compact = compact.sort_values(["roc_auc", "pr_auc"], ascending=False).reset_index(drop=True)

manuscript_cols = [
    "analysis_stage", "feature_set", "model",
    "roc_auc", "pr_auc", "balanced_accuracy",
    "sensitivity", "specificity", "precision", "f1",
    "threshold", "n_test", "n_predictors"
]
manuscript_cols = [c for c in manuscript_cols if c in compact.columns]

manuscript_table = compact[manuscript_cols].head(12).copy()
display(manuscript_table)

manuscript_table.to_csv(OUT_DIR / "05_manuscript_ready_model_performance_table.csv", index=False)


## Code — Figures

In [ ]:
# ============================================================
# 09. Figure 1 — ROC-AUC by top models
# ============================================================

fig_df = manuscript_table.copy().head(10)

if not fig_df.empty and "roc_auc" in fig_df.columns:
    labels = (
        fig_df["analysis_stage"].astype(str).str.replace(" models", "", regex=False)
        + "\n" + fig_df["model"].astype(str)
    )

    plt.figure(figsize=(10, max(5, 0.55 * len(fig_df))))
    plt.barh(range(len(fig_df)), fig_df["roc_auc"].values)
    plt.yticks(range(len(fig_df)), labels)
    plt.xlabel("Held-out test ROC-AUC")
    plt.title("Top model performance by ROC-AUC")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_01_top_models_roc_auc.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# 10. Figure 2 — PR-AUC by top models
# ============================================================

if not fig_df.empty and "pr_auc" in fig_df.columns:
    labels = (
        fig_df["analysis_stage"].astype(str).str.replace(" models", "", regex=False)
        + "\n" + fig_df["model"].astype(str)
    )

    plt.figure(figsize=(10, max(5, 0.55 * len(fig_df))))
    plt.barh(range(len(fig_df)), fig_df["pr_auc"].values)
    plt.yticks(range(len(fig_df)), labels)
    plt.xlabel("Held-out test PR-AUC")
    plt.title("Top model performance by PR-AUC")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_02_top_models_pr_auc.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# 11. Figure 3 — Incremental performance deltas
# ============================================================

if not incremental_summary.empty:
    delta_cols = [
        "delta_test_roc_auc",
        "delta_test_pr_auc",
        "delta_test_balanced_accuracy",
        "delta_test_sensitivity",
        "delta_test_specificity",
        "delta_test_f1"
    ]
    available_delta_cols = [c for c in delta_cols if c in incremental_summary.columns]

    plot_df = incremental_summary[["increment_type"] + available_delta_cols].copy()
    long_df = plot_df.melt(id_vars="increment_type", var_name="metric", value_name="delta")

    # Plot each metric as grouped bars using simple positions
    plt.figure(figsize=(11, 6))
    x = np.arange(len(long_df))
    plt.bar(x, long_df["delta"].values)
    plt.xticks(x, long_df["metric"].str.replace("delta_test_", "", regex=False), rotation=45, ha="right")
    plt.axhline(0, linewidth=1)
    plt.ylabel("Delta in held-out test performance")
    plt.title("Incremental performance changes from added data modalities")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_03_incremental_performance_deltas.png", dpi=300, bbox_inches="tight")
    plt.show()


## Quality Control Checklist

In [ ]:
# ============================================================
# 12. Quality control checklist
# ============================================================

qc = []

qc.append({
    "qc_item": "Notebook 05/06/09/11 outputs located",
    "status": "PASS" if any([NB05_DIR.exists(), NB06_DIR.exists(), NB09_DIR.exists(), NB11_DIR.exists()]) else "FAIL",
    "details": "At least one previous modeling output directory was found."
})

qc.append({
    "qc_item": "Consolidated test table created",
    "status": "PASS" if not all_test_clean.empty else "FAIL",
    "details": f"rows={len(all_test_clean)}"
})

qc.append({
    "qc_item": "Incremental summary created",
    "status": "PASS" if not incremental_summary.empty else "WARN",
    "details": f"rows={len(incremental_summary)}"
})

qc.append({
    "qc_item": "Final decision table created",
    "status": "PASS" if len(final_decision) > 0 else "FAIL",
    "details": f"rows={len(final_decision)}"
})

qc.append({
    "qc_item": "Manuscript performance table created",
    "status": "PASS" if len(manuscript_table) > 0 else "FAIL",
    "details": f"rows={len(manuscript_table)}"
})

qc.append({
    "qc_item": "Figures saved",
    "status": "PASS" if len(list(FIG_DIR.glob('*.png'))) > 0 else "WARN",
    "details": f"figures={len(list(FIG_DIR.glob('*.png')))}"
})

qc_df = pd.DataFrame(qc)
display(qc_df)

qc_df.to_csv(OUT_DIR / "06_quality_control_checklist.csv", index=False)


## Scientific Interpretation

In [ ]:
# ============================================================
# 13. Summary report
# ============================================================

best_by_roc = top_models_by_metric[top_models_by_metric["selection_metric"] == "roc_auc"] if not top_models_by_metric.empty else pd.DataFrame()
best_by_pr = top_models_by_metric[top_models_by_metric["selection_metric"] == "pr_auc"] if not top_models_by_metric.empty else pd.DataFrame()

report_lines = []
report_lines.append("Notebook 12 — Final Model Selection and Manuscript-Ready Performance Summary")
report_lines.append("=" * 80)
report_lines.append(f"Project folder: {PROJECT_DIR}")
report_lines.append(f"Output folder: {OUT_DIR}")
report_lines.append("")
report_lines.append("Main interpretation:")
report_lines.append("- The prediction task remains challenging; overall discrimination is modest.")
report_lines.append("- Added DaTSCAN/SBR and SAA features do not produce a sufficiently strong or stable improvement to justify a final high-performance clinical prediction claim.")
report_lines.append("- The most defensible manuscript framing is a transparent reproducible benchmark of clinical, imaging, and SAA biomarker predictors for motor progression.")
report_lines.append("")
if not best_by_roc.empty:
    r = best_by_roc.iloc[0]
    report_lines.append("Best held-out ROC-AUC row:")
    report_lines.append(f"- Analysis stage: {r.get('analysis_stage', 'NA')}")
    report_lines.append(f"- Feature set: {r.get('feature_set', 'NA')}")
    report_lines.append(f"- Model: {r.get('model', 'NA')}")
    report_lines.append(f"- ROC-AUC: {r.get('roc_auc', np.nan):.4f}")
    report_lines.append(f"- PR-AUC: {r.get('pr_auc', np.nan):.4f}")
    report_lines.append("")
if not best_by_pr.empty:
    r = best_by_pr.iloc[0]
    report_lines.append("Best held-out PR-AUC row:")
    report_lines.append(f"- Analysis stage: {r.get('analysis_stage', 'NA')}")
    report_lines.append(f"- Feature set: {r.get('feature_set', 'NA')}")
    report_lines.append(f"- Model: {r.get('model', 'NA')}")
    report_lines.append(f"- ROC-AUC: {r.get('roc_auc', np.nan):.4f}")
    report_lines.append(f"- PR-AUC: {r.get('pr_auc', np.nan):.4f}")
    report_lines.append("")
report_lines.append("Recommended next step:")
report_lines.append("- Move to Notebook 13: manuscript-ready exploratory analysis, cohort description, and reporting checklist.")
report_lines.append("- Avoid further repeated tuning on the same held-out test set.")

report = "\n".join(report_lines)
print(report)

with open(OUT_DIR / "07_notebook_12_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(report)


## Expected Output

The key files saved by this notebook are:

- `01_consolidated_test_performance_all_models.csv`
- `02_incremental_performance_summary.csv`
- `03_top_models_by_metric.csv`
- `04_final_model_selection_decision_table.csv`
- `05_manuscript_ready_model_performance_table.csv`
- `06_quality_control_checklist.csv`
- `07_notebook_12_summary_report.txt`
- `figures/figure_01_top_models_roc_auc.png`
- `figures/figure_02_top_models_pr_auc.png`
- `figures/figure_03_incremental_performance_deltas.png`
